# Project 6: RNN from Scratch — Character-Level Language Model

**Chapters:** 8 & 9 — RNNs, LSTMs, Sequence Models

**Goal:** Build a character-level RNN that learns to generate text. Understand hidden states, BPTT, and why vanilla RNNs struggle with long sequences.

**What you'll build:**
1. Vanilla RNN cell (forward + backward)
2. Character-level language model on a small corpus
3. Text generation by sampling from the model
4. LSTM cell to compare against vanilla RNN

**Rules:** No PyTorch/TF. NumPy only.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['font.size'] = 12

## 1. Prepare Text Data

In [ ]:
# Small corpus — enough to learn patterns, small enough for CPU training
text = """machine learning is the study of algorithms that improve through experience.
a perceptron is the simplest neural network. it classifies inputs by computing a weighted sum.
gradient descent is an optimization algorithm. it moves in the direction of steepest descent.
neural networks learn representations. each layer transforms data into a more useful form.
recurrent neural networks process sequences. they maintain a hidden state across time steps.
the hidden state is a summary of past inputs. it allows the network to have memory.
long short term memory networks solve the vanishing gradient problem.
attention mechanisms allow models to focus on relevant parts of the input.
""" * 10  # repeat for more training data

chars = sorted(set(text))
vocab_size = len(chars)
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for c, i in char_to_idx.items()}

data = np.array([char_to_idx[c] for c in text])
print(f"Corpus: {len(text)} chars, {vocab_size} unique")
print(f"Vocabulary: {''.join(chars)}")

## 2. Vanilla RNN Cell — Your Turn

### The math

At each time step $t$:

$$\mathbf{h}_t = \tanh(W_{hh} \mathbf{h}_{t-1} + W_{xh} \mathbf{x}_t + \mathbf{b}_h)$$
$$\mathbf{o}_t = W_{hy} \mathbf{h}_t + \mathbf{b}_y$$
$$\hat{\mathbf{y}}_t = \text{softmax}(\mathbf{o}_t)$$

Loss: cross-entropy per time step.

In [ ]:
def softmax(z):
    """Numerically stable softmax."""
    e = np.exp(z - np.max(z))
    return e / e.sum()

def one_hot(idx, size):
    v = np.zeros(size)
    v[idx] = 1.0
    return v

In [ ]:
class CharRNN:
    def __init__(self, vocab_size, hidden_size=64, lr=0.01):
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.lr = lr
        
        # Weight initialization (Xavier)
        scale_xh = np.sqrt(1.0 / vocab_size)
        scale_hh = np.sqrt(1.0 / hidden_size)
        self.Wxh = np.random.randn(hidden_size, vocab_size) * scale_xh
        self.Whh = np.random.randn(hidden_size, hidden_size) * scale_hh
        self.Why = np.random.randn(vocab_size, hidden_size) * scale_hh
        self.bh = np.zeros(hidden_size)
        self.by = np.zeros(vocab_size)
        self.loss_history = []
    
    def forward(self, inputs, h_prev):
        """Forward pass through a sequence of character indices.
        
        Args:
            inputs: list of character indices (length T)
            h_prev: previous hidden state, shape (hidden_size,)
        Returns:
            probs: dict mapping time step to softmax output
            hs: dict mapping time step to hidden state
            xs: dict mapping time step to one-hot input
        """
        xs, hs, probs = {}, {}, {}
        hs[-1] = h_prev.copy()
        
        for t, idx in enumerate(inputs):
            ### YOUR CODE ###
            # xs[t] = one_hot encoding of idx
            # hs[t] = tanh(Wxh @ xs[t] + Whh @ hs[t-1] + bh)
            # probs[t] = softmax(Why @ hs[t] + by)
            raise NotImplementedError("RNN forward step")
        
        return probs, hs, xs
    
    def loss(self, probs, targets):
        """Cross-entropy loss over the sequence."""
        ### YOUR CODE ###
        # sum of -log(probs[t][targets[t]]) for each t
        raise NotImplementedError
    
    def backward(self, probs, hs, xs, targets):
        """BPTT: compute gradients through time.
        
        Returns:
            grads: dict with dWxh, dWhh, dWhy, dbh, dby
        """
        dWxh = np.zeros_like(self.Wxh)
        dWhh = np.zeros_like(self.Whh)
        dWhy = np.zeros_like(self.Why)
        dbh = np.zeros_like(self.bh)
        dby = np.zeros_like(self.by)
        dh_next = np.zeros(self.hidden_size)
        
        for t in reversed(range(len(targets))):
            ### YOUR CODE ###
            # dy = probs[t].copy(); dy[targets[t]] -= 1  (softmax + CE gradient)
            # dWhy += dy (outer) hs[t]
            # dby += dy
            # dh = Why.T @ dy + dh_next
            # dh_raw = (1 - hs[t]**2) * dh  (tanh derivative)
            # dWxh += dh_raw (outer) xs[t]
            # dWhh += dh_raw (outer) hs[t-1]
            # dbh += dh_raw
            # dh_next = Whh.T @ dh_raw
            raise NotImplementedError("BPTT")
        
        # Clip gradients to prevent explosion
        for grad in [dWxh, dWhh, dWhy, dbh, dby]:
            np.clip(grad, -5, 5, out=grad)
        
        return {'dWxh': dWxh, 'dWhh': dWhh, 'dWhy': dWhy, 'dbh': dbh, 'dby': dby}
    
    def update(self, grads):
        """SGD update. (Done for you.)"""
        self.Wxh -= self.lr * grads['dWxh']
        self.Whh -= self.lr * grads['dWhh']
        self.Why -= self.lr * grads['dWhy']
        self.bh -= self.lr * grads['dbh']
        self.by -= self.lr * grads['dby']
    
    def sample(self, seed_char, length=100, temperature=1.0):
        """Generate text by sampling from the model. (Done for you.)"""
        h = np.zeros(self.hidden_size)
        idx = char_to_idx[seed_char]
        result = [seed_char]
        
        for _ in range(length):
            x = one_hot(idx, self.vocab_size)
            h = np.tanh(self.Wxh @ x + self.Whh @ h + self.bh)
            o = self.Why @ h + self.by
            o = o / temperature
            p = softmax(o)
            idx = np.random.choice(self.vocab_size, p=p)
            result.append(idx_to_char[idx])
        
        return ''.join(result)

### Sanity checks

In [ ]:
# Test forward pass shapes
rnn_test = CharRNN(vocab_size, hidden_size=16, lr=0.01)
h0 = np.zeros(16)
test_inputs = [0, 1, 2]
test_targets = [1, 2, 3]
probs_t, hs_t, xs_t = rnn_test.forward(test_inputs, h0)

assert len(probs_t) == 3, f"Expected 3 time steps, got {len(probs_t)}"
assert probs_t[0].shape == (vocab_size,), f"Probs shape should be ({vocab_size},), got {probs_t[0].shape}"
assert abs(probs_t[0].sum() - 1.0) < 1e-6, "Softmax output must sum to 1"
assert hs_t[0].shape == (16,), f"Hidden state shape should be (16,), got {hs_t[0].shape}"
print("OK: Forward pass shapes correct")

# Test loss is positive
loss_t = rnn_test.loss(probs_t, test_targets)
assert loss_t > 0, f"Loss should be positive, got {loss_t}"
print(f"OK: Loss = {loss_t:.4f} (positive)")

# Test backward returns correct gradient keys
grads_t = rnn_test.backward(probs_t, hs_t, xs_t, test_targets)
assert set(grads_t.keys()) == {'dWxh', 'dWhh', 'dWhy', 'dbh', 'dby'}, f"Wrong gradient keys: {grads_t.keys()}"
assert grads_t['dWxh'].shape == rnn_test.Wxh.shape, "dWxh shape mismatch"
print("OK: Backward pass returns correct gradients")

# Test that training reduces loss
loss_before = rnn_test.loss(rnn_test.forward(test_inputs, h0)[0], test_targets)
for _ in range(50):
    p, h, x = rnn_test.forward(test_inputs, h0)
    g = rnn_test.backward(p, h, x, test_targets)
    rnn_test.update(g)
loss_after = rnn_test.loss(rnn_test.forward(test_inputs, h0)[0], test_targets)
assert loss_after < loss_before, f"Loss should decrease: {loss_before:.4f} -> {loss_after:.4f}"
print(f"OK: Loss decreased: {loss_before:.4f} -> {loss_after:.4f}")

print("\nAll checks passed!")

## 3. Train

In [ ]:
rnn = CharRNN(vocab_size, hidden_size=64, lr=0.01)
seq_len = 25
smooth_loss = -np.log(1.0 / vocab_size) * seq_len  # initial loss estimate

for step in range(5001):
    # Sample a random chunk
    start = np.random.randint(0, len(data) - seq_len - 1)
    inputs = data[start:start+seq_len].tolist()
    targets = data[start+1:start+seq_len+1].tolist()
    
    h_prev = np.zeros(rnn.hidden_size)
    probs, hs, xs = rnn.forward(inputs, h_prev)
    loss = rnn.loss(probs, targets)
    grads = rnn.backward(probs, hs, xs, targets)
    rnn.update(grads)
    
    smooth_loss = 0.999 * smooth_loss + 0.001 * loss
    rnn.loss_history.append(smooth_loss)
    
    if step % 1000 == 0:
        print(f"Step {step:5d}  |  loss: {smooth_loss:.2f}")
        print(f"  Sample: {rnn.sample('t', length=80)}")
        print()

In [ ]:
plt.plot(rnn.loss_history)
plt.xlabel('Step'); plt.ylabel('Smoothed Loss'); plt.title('RNN Training')
plt.grid(True, alpha=0.3); plt.show()

## 4. Experiments

### A: Temperature
Generate text with temperature 0.5 (confident), 1.0 (normal), 2.0 (creative). What changes?

In [ ]:
for t in [0.3, 0.7, 1.0, 1.5]:
    print(f"Temperature {t}: {rnn.sample('t', length=100, temperature=t)}")
    print()

**Your observation:** *(How does temperature affect diversity vs coherence?)*

...

### B: Hidden size
Try hidden_size = 16, 64, 128. How does capacity affect learning?

In [ ]:
# Your experiment here


## 5. What You Learned

1. What information does the hidden state carry?
2. Why do gradients vanish in long sequences? (Think about repeated multiplication by $W_{hh}$.)
3. How does the LSTM solve this? (Hint: the cell state is additive, not multiplicative.)
4. Why does temperature work? (Think about softmax with scaled logits.)

---
**Next project:** CNN — apply neural networks to images.